# Marketing Agent — Data & Preferences (Phases 1-3)

Build the campaign metrics, generate strong/weak recommendation candidates, and judge them
into a DPO preference dataset. **No GPU needed** — this is all Gemini API + pandas.

Run this if you'd rather not set up a local Python 3.10+ venv. Output is
`data/preferences/preferences.jsonl`, which you then commit/push and consume in the training notebook.


## 1. Clone + install the local pipeline deps

In [ ]:
REPO_URL = 'https://github.com/SalahnAI/Preference-Optimization-for-Marketing-Agents.git'
import os
if not os.path.isdir('Preference-Optimization-for-Marketing-Agents'):
    !git clone $REPO_URL
%cd Preference-Optimization-for-Marketing-Agents
!pip install -q -r requirements.txt

## 2. Gemini API key

Get a free key at https://aistudio.google.com/apikey

In [ ]:
import getpass, os
os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')
os.environ['PYTHONPATH'] = 'src'

## 3. Phase 1 — campaign metrics

`--source synthetic` (default) or `--source criteo` if you uploaded `data/raw/train.txt`.

In [ ]:
!PYTHONPATH=src python -m marketing_agent.data_prep --source synthetic --n 400

## 4. Phase 2 — generate strong vs weak candidates

Two candidates per campaign. Cached on disk, so reruns are cheap.

In [ ]:
!PYTHONPATH=src python -m marketing_agent.generate

## 5. Phase 3 — judge pairs into the DPO dataset

In [ ]:
!PYTHONPATH=src python -m marketing_agent.build_preferences

In [ ]:
# Inspect the result
import json
rows = [json.loads(l) for l in open('data/preferences/preferences.jsonl')]
print(f'{len(rows)} preference pairs')
print('\nPROMPT:\n', rows[0]['prompt'][:300])
print('\n=== CHOSEN ===\n', rows[0]['chosen'][:400])
print('\n=== REJECTED ===\n', rows[0]['rejected'][:400])
print('\njudge_reason:', rows[0].get('judge_reason'))

## 6. Commit the dataset back to the repo

`preferences.jsonl` is small and *not* gitignored — it's the core artifact. Push it so the
training notebook can pull it. (Set up a GitHub token / `git config` first.)

In [ ]:
# Example — fill in your identity, then uncomment:
# !git config user.email 'you@example.com'
# !git config user.name 'Your Name'
# !git add data/preferences/preferences.jsonl
# !git commit -m 'Add preference dataset'
# !git push

# Or just download it:
from google.colab import files
files.download('data/preferences/preferences.jsonl')